# AdaBoost（1995）

- **核心原理**：加權弱分類器
- **變體**：Discrete AdaBoost（Freund & Schapire）、SAMME（多類別；sklearn 現在的版本）、Real AdaBoost；理論上等於「用指數損失做前向分段加法模型」→ 第 12 章 GBDT 的前身
- **推薦資料集**：Breast Cancer Wisconsin（569 筆、30 個特徵；**0 = 惡性 malignant、1 = 良性 benign**；跟第三章同一份資料）

本章依照 README 的「每章標準流程」：EDA → Baseline → 手寫模型 → 套件實作 → 調參 → 測試集評估 → 診斷 → 本章小結。

第十章的 Random Forest 是 **bagging**：很多棵「強而不穩」的樹**平行**地各自長，再平均 → 降低**變異**。
AdaBoost 是 **boosting**：很多個「弱而穩」的分類器（通常是只切一刀的 **stump**）**一個接一個**地學，每一個專門處理前面的人做錯的樣本 → 降低**偏差**。

做法只有兩個動作，重複 $T$ 次：

1. 在**加權**的資料上訓練一個弱分類器；
2. 被它分錯的樣本**加重權重**，下一個分類器就會更在乎它們。

最後的預測是所有弱分類器的**加權投票**，表現越好的分類器票越重。

這一章的重點：**權重怎麼更新、每個分類器的票數 $\alpha$ 從哪裡來（指數損失）、為什麼訓練誤差到 0 之後測試誤差還會降（margin）、為什麼 AdaBoost 怕標錯的資料。**

### 閱讀指南：本章的符號與跨章引用

下一格是「前置複習」；第 2、3A、3B、3C、4、5、7 節開頭另有各自的符號表或名詞表。

**資料**：30 個特徵是細胞核影像量出來的 10 種量（半徑、紋理、周長、面積、平滑度……）× 3 種統計（mean、error、worst）。目標 0 = 惡性、1 = 良性。

**本章最常出現的符號**（各小節開頭有完整符號表）：

| 符號 | 意義 | 本章的值 |
| --- | --- | --- |
| $n$、$d$ | 訓練樣本數、特徵數 | 455、30（以載入結果為準） |
| $y_i$ | AdaBoost 內部用的 $\pm 1$ 標籤：良性 = +1、惡性 = −1 | — |
| $t$、$T$ | 第 $t$ 輪、總輪數（`n_estimators`） | — |
| $w_i^{(t)}$ | 第 $t$ 輪時第 $i$ 筆樣本的權重，$\sum_i w_i = 1$ | $(n,)$ |
| $h_t(x)$ | 第 $t$ 個弱分類器（stump），輸出 $\pm 1$ | — |
| $\varepsilon_t$ | $h_t$ 的**加權**錯誤率 $\sum_i w_i [h_t(x_i) \ne y_i]$ | — |
| $\alpha_t$ | $h_t$ 的票數 | — |
| $F_T(x)$ | 加權投票的總分 $\sum_{t=1}^{T} \alpha_t h_t(x)$；預測 = $\text{sign}(F_T(x))$ | — |
| margin | 正規化的分數 $y_i F_T(x_i) / \sum_t \alpha_t \in [-1, 1]$：> 0 分對，越大越「有把握」 | — |
| stump | 深度 1 的決策樹：一個特徵、一個門檻、兩片葉子 | — |

**跨章引用對照**：

| 文中寫法 | 指的是 | 那裡的結論（重述） |
| --- | --- | --- |
| 第三章 | 第一部分第 3 節：邏輯回歸（`01-classic-ml/03-logistic-regression`），同一份 Breast Cancer | 良性 62.6%、惡性 37.4%；30 個特徵尺度差 $10^5$ 倍、高度共線；測試集 114 筆，邏輯回歸 test accuracy 0.9825、AUC 約 0.995；`recall_malignant` = 惡性被抓到的比例。 |
| 第三章 log loss | 同上 | $\log(1 + e^{-y f})$（$\pm 1$ 的寫法）；本章的指數損失 $e^{-y f}$ 是它的「近親」。 |
| 第五章 | 決策樹（`01-classic-ml/05-decision-tree`） | 找最好的一刀：排序 + `cumsum` 一次算出所有切點左邊的類別筆數；Gini 下降最大的那一刀。深度 1 的樹叫 stump。 |
| 第五章「單調轉換」 | 同上 | 樹只看排序，不需要標準化。 |
| 第八章 | SVM | margin $y f(x)$：> 0 分對、越大越穩；最大化 margin 有助於泛化。hinge loss 只看 margin < 1 的點。 |
| 第八章的重複 CV | 同上 | 資料少（這裡 455 筆）時用 `RepeatedStratifiedKFold`，平均才穩。 |
| 第十章 | Random Forest | bagging：平行地長很多棵長滿的樹再平均，降低**變異**；平均不會降低偏差。 |

### 前置複習：本章會用到的舊知識

#### 數學

**1. 加權平均與加權錯誤率**：權重 $w_i \ge 0$、$\sum_i w_i = 1$ 時，加權錯誤率 $\varepsilon = \sum_i w_i [\text{錯}_i]$。例：3 筆、權重 $(0.5, 0.25, 0.25)$，第 1、3 筆錯 → $\varepsilon = 0.5 + 0.25 = 0.75$。所有權重相等（$1/n$）時就是一般的錯誤率。

**2. 指數與對數**：$e^{a + b} = e^a e^b$、$\ln(e^a) = a$、$\frac{d}{da} e^{ca} = c\,e^{ca}$。$\ln\frac{1 - \varepsilon}{\varepsilon}$：$\varepsilon = 0.5$ 時是 0、$\varepsilon \to 0$ 時 $\to \infty$、$\varepsilon > 0.5$ 時是負的。

**3. 一元函數的最小值**：令導數 = 0。例：$g(a) = A e^{-a} + B e^{a}$（$A, B > 0$）→ $g'(a) = -A e^{-a} + B e^a = 0$ → $e^{2a} = A / B$ → $a = \frac{1}{2}\ln\frac{A}{B}$。（TODO 8 會用這個推出 $\alpha$。）

**4. 指數損失 vs 0-1 損失**：對 margin $z = y f(x)$，0-1 損失 $[z < 0]$；指數損失 $e^{-z}$ 永遠 $\ge [z < 0]$（是它的上界），而且 $z$ 越大越小 —— **分對了還會繼續把 margin 推大**。

**5. 正規化**：$w \leftarrow w / \sum_i w_i$ 讓總和回到 1。乘上一個共同的常數再正規化，結果不變。

**6. 偏差—變異**（重述）：stump 只能切一刀 → 偏差很高、變異很低。Bagging 平均「低偏差高變異」的模型；boosting 把「高偏差低變異」的模型一個個**加起來**，降低偏差。

#### 程式

**7. 加權的累加**（第五章 `cumsum` 的延伸）：
```python
onehot = np.eye(2)[y_idx]            # (n, 2)：每筆在自己類別那格是 1
W = onehot * w[:, None]              # (n, 2)：改成「自己的權重」
np.cumsum(W[order], axis=0)          # 排序後，前 i+1 筆各類的「權重總和」
```
所有 $w_i = 1$ 時就是第五章的筆數。

**8. `zip` 平行走兩個串列**：`for h, a in zip(self.stumps_, self.alphas_):` 一次拿到第 $t$ 個 stump 與它的 $\alpha$。

**9. 累加出每一輪的分數**：`H` 是 $(T, n)$（每一輪、每一筆的 $h_t(x_i)$），`np.cumsum(alphas[:, None] * H, axis=0)` 的第 $t$ 列 = 前 $t+1$ 輪的總分 $F_{t+1}$。

**10. `np.where(cond, a, b)`、布林遮罩 `miss = h != y`**；`w[miss].sum()` = 錯的那些的權重總和。

---
## 1. 資料與探索性資料分析（EDA）

第三章已經詳細看過這份資料，這裡只看跟 AdaBoost 有關的部分：**每個特徵單獨切一刀能做到多好**（stump 的能力），以及哪些樣本「難」。

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from learning_map.data import load_breast_cancer

X_train, X_test, y_train, y_test = load_breast_cancer()
y_train = pd.Series(np.asarray(y_train), index=X_train.index, name="target").astype(int)
y_test = pd.Series(np.asarray(y_test), index=X_test.index, name="target").astype(int)
Xtr, Xte = X_train.to_numpy(dtype=float), X_test.to_numpy(dtype=float)
ytr, yte = y_train.to_numpy(), y_test.to_numpy()
FEATURES = X_train.columns.tolist()

print("train / test：", X_train.shape, X_test.shape)
display(pd.DataFrame({"train": y_train.value_counts(normalize=True), "test": y_test.value_counts(normalize=True)}).rename(index={0: "malignant (0)", 1: "benign (1)"}).round(3))
print(f"測試集 {len(yte)} 筆：1 筆 = {1 / len(yte):.2%}")

In [ ]:
# 每個特徵「只切一刀」的最好 accuracy（在訓練集上，暴力試所有門檻）
# 觀察：（最好的單一特徵是哪個、accuracy 多少？最差的呢？有多少個特徵單獨就超過 0.85？
#        「弱分類器」的定義是比亂猜好一點點；這些 stump 夠「弱」嗎？）
def best_single_cut(x: np.ndarray, y: np.ndarray) -> tuple[float, float]:
    """一個特徵上所有門檻的最好 accuracy（兩個方向都試）→ (accuracy, 門檻)。"""
    vals = np.unique(x)
    best = (0.0, np.nan)
    for t in (vals[:-1] + vals[1:]) / 2:
        acc = np.mean((x > t).astype(int) == y)
        best = max(best, (max(acc, 1 - acc), t))
    return best


stump_acc = pd.DataFrame([best_single_cut(Xtr[:, j], ytr) for j in range(Xtr.shape[1])], index=FEATURES, columns=["best stump acc", "threshold"])
stump_acc = stump_acc.sort_values("best stump acc", ascending=False)
display(stump_acc.head(8).round(4))
print("最差的 3 個：", stump_acc.tail(3)["best stump acc"].round(3).to_dict())
print("單獨超過 0.85 的特徵數：", int((stump_acc["best stump acc"] > 0.85).sum()), "/", len(FEATURES))

In [ ]:
# 兩個最好的特徵的散點圖：一刀切得開嗎？
# 觀察：（兩類在哪裡重疊？靠一條「垂直或水平」的線（stump）切，錯的會是哪些點？兩刀、三刀呢？）
f1, f2 = stump_acc.index[:2]
fig, ax = plt.subplots(figsize=(7, 5))
for lab, name in [(0, "malignant"), (1, "benign")]:
    m = ytr == lab
    ax.scatter(X_train.loc[m, f1], X_train.loc[m, f2], s=14, label=name, alpha=0.7)
ax.axvline(stump_acc.loc[f1, "threshold"], color="k", ls="--", lw=1, label=f"best stump on {f1}")
ax.set_xlabel(f1)
ax.set_ylabel(f2)
ax.legend()
plt.show()

In [ ]:
# TODO 1（問答）EDA 小結
#
# (1) 單一 stump 最好能到多少 accuracy？AdaBoost 要求每個弱分類器「比亂猜好」（加權錯誤率 < 0.5）。
#     第一輪（所有權重相等）一定做得到嗎？之後權重集中在「難」的點上，還做得到嗎？
# 答：
#
# (2) 散點圖上，一刀切不開的點在哪裡？如果「第二個 stump 只看這些點」，它會切在哪個方向？
#     兩個 stump 怎麼「加起來」變成一條斜線或折線？（提示：F(x) = α₁h₁(x) + α₂h₂(x)，每個 h 都是 ±1 的階梯）
# 答：
#
# (3) 第三章花了很多力氣在標準化。AdaBoost + stump 需要標準化嗎？（第五章「單調轉換」）
# 答：

---
## 2. Baseline

- **Dummy prior**：猜多數（良性）。
- **單一 stump**：AdaBoost 的第 1 輪就是它（權重全部相等時）。
- **深度 3 的決策樹**：第五章最好的那種「小樹」。

#### 本節名詞

- **`RepeatedStratifiedKFold(5, 10)`**：資料少（455 筆），重複 10 次 5-fold（第八章）。
- **`recall_malignant`**：惡性被抓到的比例（第三章用來看「漏診」的指標）；惡性是 0，所以要用 `pos_label=0`。

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=42)
SCORING = {"accuracy": "accuracy", "roc_auc": "roc_auc", "recall_malignant": make_scorer(recall_score, pos_label=0)}


def cv_summary(model, X=X_train, y=y_train) -> dict[str, str]:
    """重複 5-fold CV：accuracy、ROC AUC、惡性 recall 的「平均 ± 標準差」。"""
    s = cross_validate(model, X, y, cv=cv, scoring=SCORING, n_jobs=-1)
    return {k: f"{s['test_' + k].mean():.4f} ± {s['test_' + k].std():.4f}" for k in SCORING}


baseline = pd.DataFrame(
    {
        "Dummy prior": cv_summary(DummyClassifier(strategy="prior")),
        "Single stump": cv_summary(DecisionTreeClassifier(max_depth=1)),
        "Tree max_depth=3": cv_summary(DecisionTreeClassifier(max_depth=3, random_state=0)),
    }
).T
display(baseline)

In [ ]:
# TODO 2（問答）
#
# (1) 單一 stump 的 accuracy 和 AUC 各多少？為什麼它的 AUC 特別低？（提示：stump 只有兩片葉子 → predict_proba 只有幾種值？第五章 TODO 16 (2)）
# 答：
#
# (2) 深度 3 的樹比 stump 好很多。那為什麼 AdaBoost 偏偏要用「最弱」的 stump，而不是直接用深一點的樹？
#     先猜，3C 的實驗 4 會回來看。（提示：偏差—變異；一個接一個加起來的東西，每一個應該是穩還是不穩？）
# 答：

---
## 3. 手寫模型

| 小節 | 做什麼 | TODO |
| --- | --- | --- |
| 3A | 加權的 stump：第五章找切點的方法，「筆數」換成「權重總和」 | 3 |
| 3B | AdaBoost：加權錯誤率 → 票數 $\alpha$ → 更新權重；加權投票 | 4、5、6、7 |
| 3C | 實驗：輪數與 margin、權重集中在哪、標錯的資料、基底分類器的深度與學習率 | — |

### 3A. 加權的 stump

AdaBoost 每一輪都要在**加權**的資料上訓練一個 stump。第五章找切點的方法幾乎不用改：Gini 只看「各類的比例」，把「各類的**筆數**」換成「各類的**權重總和**」就好。

$$p_k(t) = \frac{\sum_{i \in t,\ y_i = k} w_i}{\sum_{i \in t} w_i}, \qquad \Delta I = I(\text{parent}) - \frac{W_L}{W}I(L) - \frac{W_R}{W}I(R)$$

$W_L$、$W_R$ 是左右兩邊的權重總和（取代第五章的 $n_L$、$n_R$）。葉子的預測：權重比較大的那一類（$\pm 1$）。

sklearn 的 `AdaBoostClassifier` 預設的基底分類器就是 `DecisionTreeClassifier(max_depth=1)`，它也是用**加權 Gini** 挑切點（不是直接最小化加權錯誤率）。手寫版照做，才能跟 sklearn 一模一樣。

#### 符號表（3A）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $x$ | 一個特徵（所有樣本） | $(n,)$ | `X[:, j]` |
| $s$ | $\pm 1$ 標籤 | $(n,)$ | `s` |
| $w$ | 樣本權重 | $(n,)$ | `w` |
| $W$ | 加權的 one-hot：第 0 欄 = 是 −1 的權重、第 1 欄 = 是 +1 的權重 | $(n, 2)$ | `W` |
| left | 排序後，切點 $i$ 左邊各類的權重總和 | $(n-1, 2)$ | `left` |
| total | 各類的權重總和 | $(2,)$ | `total` |

**下標公式**（TODO 3）：$\text{left}_{ik} = \sum_{r \le i} W_{\text{order}[r], k}$ —— 跟第五章 TODO 4 一樣，只是 one-hot 換成加權的 one-hot。

#### 最小例子：TODO 3（加權的累加）

4 筆、一個特徵：$x = (3, 1, 2, 4)$、$s = (+1, -1, +1, -1)$、$w = (0.1, 0.2, 0.3, 0.4)$。

1. **正確答案**：排序後，迴圈逐個切點加總左邊各類的權重。排序後 $x = (1, 2, 3, 4)$、$s = (-1, +1, +1, -1)$、$w = (0.2, 0.3, 0.1, 0.4)$ → left = `[[0.2, 0], [0.2, 0.3], [0.2, 0.4]]`。
2. **積木**：`np.stack([w * (s < 0), w * (s > 0)], axis=1)`、`W[order]`、第五章的 `np.cumsum(..., axis=0)[:-1]`。
3. **留給你**：組出 `left`。

In [ ]:
# 最小例子：TODO 3（加權的累加）
x_mini = np.array([3.0, 1.0, 2.0, 4.0])
s_mini = np.array([1, -1, 1, -1])
w_mini = np.array([0.1, 0.2, 0.3, 0.4])

# ---- 1. 正確答案：迴圈 ----
order_mini = np.argsort(x_mini, kind="stable")
left_loop = []
for i in range(3):
    first = order_mini[: i + 1]
    left_loop.append([round(float(w_mini[first][s_mini[first] < 0].sum()), 4), round(float(w_mini[first][s_mini[first] > 0].sum()), 4)])
print("正確答案 left =", left_loop)

# ---- 2. 積木 ----
W_mini = np.stack([w_mini * (s_mini < 0), w_mini * (s_mini > 0)], axis=1)
print("\nW = [w·[s<0], w·[s>0]]", W_mini.shape, "\n", W_mini)
print("W[order]               \n", W_mini[order_mini])
print("所有 w = 1 時 W 就是第五章的 one-hot：", np.stack([(s_mini < 0) * 1.0, (s_mini > 0) * 1.0], axis=1).tolist())

# ---- 3. 留給你 ----
# 問題：left = np.cumsum(?, axis=?)[?]
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(WeightedStump()._left_weights(x_mini, s_mini, w_mini)[1].round(4))

In [ ]:
from typing import Self

from numpy.typing import ArrayLike


def gini(c: np.ndarray) -> np.ndarray:
    """(..., 2) 的「各類權重」→ Gini；全 0 的列給 0。"""
    n = c.sum(axis=-1, keepdims=True)
    p = np.divide(c, n, out=np.zeros_like(c), where=n > 0)
    return (p * (1 - p)).sum(axis=-1)


class WeightedStump:
    """加權 Gini 挑切點的 stump。標籤 ±1；fit 後有 feature_、threshold_、left_value_、right_value_（都是 ±1）。"""

    def _left_weights(self, x: np.ndarray, s: np.ndarray, w: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """回傳 (排序後的 x (n,), left (n−1, 2))：切點 i 左邊「是 −1 的權重、是 +1 的權重」。"""
        order = np.argsort(x, kind="stable")
        W = np.stack([w * (s < 0), w * (s > 0)], axis=1)  # (n, 2) 加權的 one-hot
        # TODO 3（程式）：排序後累加，去掉最後一列
        raise NotImplementedError("TODO 3：加權累加")
        return x[order], left

    def fit(self, X: np.ndarray, s: np.ndarray, w: np.ndarray) -> Self:
        total = np.array([w[s < 0].sum(), w[s > 0].sum()])
        best_gain = -np.inf
        for j in range(X.shape[1]):
            xs, left = self._left_weights(X[:, j], s, w)
            right = total - left
            WL, WR = left.sum(axis=1), right.sum(axis=1)
            gain = gini(total) - (WL * gini(left) + WR * gini(right)) / total.sum()
            valid = xs[:-1] < xs[1:]
            if not valid.any():
                continue
            i = int(np.argmax(np.where(valid, gain, -np.inf)))
            if gain[i] > best_gain + 1e-12:
                best_gain = gain[i]
                self.feature_, self.threshold_ = j, (xs[i] + xs[i + 1]) / 2
                self.left_value_ = 1 if left[i, 1] > left[i, 0] else -1
                self.right_value_ = 1 if right[i, 1] > right[i, 0] else -1
        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        """(n,) 的 ±1。"""
        return np.where(X[:, self.feature_] <= self.threshold_, self.left_value_, self.right_value_)


s_tr = np.where(ytr == 1, 1, -1)  # 良性 +1、惡性 −1
st = WeightedStump().fit(Xtr, s_tr, np.full(len(ytr), 1 / len(ytr)))
sk_st = DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr)
print(f"手寫 stump：{FEATURES[st.feature_]} <= {st.threshold_:.4f} → 左 {st.left_value_:+d}、右 {st.right_value_:+d}")
print(f"sklearn   ：{FEATURES[sk_st.tree_.feature[0]]} <= {sk_st.tree_.threshold[0]:.4f}")
print("✅ 等權重時預測與 sklearn stump 一致比例：", np.mean((st.predict(Xte) > 0).astype(int) == sk_st.predict(Xte)), "（應 = 1.0）")
w_rand = np.random.default_rng(0).random(len(ytr))
st_w = WeightedStump().fit(Xtr, s_tr, w_rand / w_rand.sum())
sk_w = DecisionTreeClassifier(max_depth=1).fit(Xtr, ytr, sample_weight=w_rand / w_rand.sum())
print("✅ 隨機權重時預測與 sklearn（sample_weight）一致比例：", np.mean((st_w.predict(Xte) > 0).astype(int) == sk_w.predict(Xte)), "（應 = 1.0）")

### 3B. AdaBoost

```
w_i = 1/n
for t = 1..T:
    h_t = 在權重 w 上訓練的 stump
    ε_t = Σ_i w_i [h_t(x_i) ≠ y_i]                       ← TODO 4
    α_t = ln((1 − ε_t) / ε_t)                            ← TODO 4
    w_i ← w_i · exp(α_t · [h_t(x_i) ≠ y_i])，再正規化     ← TODO 5
F_T(x) = Σ_t α_t h_t(x)，預測 sign(F_T(x))                ← TODO 6
```

- **$\alpha_t$**：$\varepsilon_t$ 越小（越準），票越重；$\varepsilon_t = 0.5$（亂猜）時 $\alpha_t = 0$。
- **權重更新**：分錯的乘上 $e^{\alpha_t} = \frac{1 - \varepsilon_t}{\varepsilon_t}$（> 1），分對的不變，再正規化。效果：**更新後，$h_t$ 在新權重下的錯誤率剛好是 0.5** —— 下一個 stump 不能再抄 $h_t$ 的答案。
- 這是 sklearn 的 **SAMME**（$K = 2$ 時）。原始論文（Discrete AdaBoost）寫成 $\alpha_t = \frac{1}{2}\ln\frac{1 - \varepsilon_t}{\varepsilon_t}$、分對的乘 $e^{-\alpha}$、分錯的乘 $e^{\alpha}$ —— 正規化之後權重完全一樣，$F$ 只差一個共同的倍數 2（不影響 sign）。TODO 8 會從指數損失推出 $\frac{1}{2}$ 的版本。
- 特殊情況：$\varepsilon_t = 0$（完全分對）→ 停止；$\varepsilon_t \ge 0.5$ → 這個分類器沒用，也停止。

#### 符號表（3B）

| 符號 | 意義 | shape | 程式裡 |
| --- | --- | --- | --- |
| $w$ | 目前的樣本權重（總和 1） | $(n,)$ | `w` |
| $h_t(X)$ | 這一輪 stump 在訓練資料上的預測 | $(n,)$ ±1 | `h` |
| miss | $h_t(x_i) \ne y_i$ | $(n,)$ bool | `miss` |
| $\varepsilon_t$、$\alpha_t$ | 加權錯誤率、票數 | 純量 | `err`、`alpha` |
| `stumps_`、`alphas_` | 每一輪的 stump 與 $\alpha$ | $T$ 個、$(T,)$ | 屬性 |
| `weights_history_` | 每一輪**開始時**的權重（實驗用） | $(T, n)$ | 屬性 |
| $H$ | 每一輪 stump 對查詢點的預測 | $(T, n_\text{query})$ | `H` |
| staged $F$ | 前 $t$ 輪的總分 | $(T, n_\text{query})$ | `staged_decision_function` |

#### 程式結構圖

```
ada = AdaBoost(n_estimators=50).fit(Xtr, ytr)
├── n_estimators、classes_ (2,)：classes_[1] ↔ +1、classes_[0] ↔ −1
├── stumps_  = [WeightedStump, WeightedStump, ...]     每個有 feature_、threshold_、left_value_、right_value_
├── alphas_  = (T,)
└── weights_history_ (T, n)

fit：for t → WeightedStump().fit(X, s, w) → miss → err、alpha (TODO 4) → 更新 w (TODO 5) → append
decision_function(X)：for stump, alpha in zip(stumps_, alphas_) 累加 alpha · stump.predict(X)   ← TODO 6
staged_decision_function(X)：H (T, n) → 依序累加                                             ← TODO 7
```

#### 最小例子：TODO 4、5（一輪 AdaBoost）

5 筆，權重都是 0.2，這一輪的 stump 錯了第 0、3 筆。

1. **正確答案**：$\varepsilon = 0.4$；$\alpha = \ln(0.6/0.4) = 0.4055$；錯的乘 $e^{\alpha} = 1.5$ → $(0.3, 0.2, 0.2, 0.3, 0.2)$ → 正規化（總和 1.2）→ $(0.25, 0.1667, 0.1667, 0.25, 0.1667)$。驗證：新權重下錯的那兩筆合計 $0.5$。
2. **積木**：`w[miss].sum()`、`np.exp(alpha * miss)`（bool 會變成 0／1）、`w / w.sum()`。
3. **留給你**：寫進 `_boost_round`。

In [ ]:
# 最小例子：TODO 4、5（一輪 AdaBoost）
w5 = np.full(5, 0.2)
miss5 = np.array([True, False, False, True, False])

# ---- 1. 正確答案：逐筆 ----
err_loop = sum(w5[i] for i in range(5) if miss5[i])
alpha_loop = np.log((1 - err_loop) / err_loop)
new_loop = [w5[i] * (np.exp(alpha_loop) if miss5[i] else 1.0) for i in range(5)]
new_loop = [v / sum(new_loop) for v in new_loop]
print(f"ε = {err_loop:.4f}、α = {alpha_loop:.4f}、e^α = {np.exp(alpha_loop):.4f}")
print("正確答案：新權重 =", np.round(new_loop, 4), "；錯的那兩筆合計", round(new_loop[0] + new_loop[3], 4))

# ---- 2. 積木 ----
print("\nw[miss].sum() =", w5[miss5].sum())
print("np.exp(alpha * miss) =", np.exp(alpha_loop * miss5).round(4), "← 分對的 e^0 = 1")
print("相乘之後總和 =", (w5 * np.exp(alpha_loop * miss5)).sum().round(4), "← 不是 1，要正規化")

# ---- 3. 留給你 ----
# 問題：_boost_round 回傳 (err, alpha, 新的 w)。
# 寫完 TODO、執行過下一格之後，取消下面的註解來對照：
# print(AdaBoost._boost_round(w5, miss5))

In [ ]:
class AdaBoost:
    """二元 AdaBoost（SAMME，K = 2），基底分類器是加權 Gini 的 stump。"""

    def __init__(self, n_estimators: int = 50) -> None:
        self.n_estimators = n_estimators

    @staticmethod
    def _boost_round(w: np.ndarray, miss: np.ndarray) -> tuple[float, float, np.ndarray]:
        """一輪的權重更新。w (n,) 總和 1、miss (n,) bool → (ε, α, 新的 w（總和 1）)。"""
        # TODO 4（程式）：加權錯誤率與票數
        #   - err = 錯的那些的權重總和 ÷ 全部權重（w 已經是總和 1，但除一下比較保險）
        #   - alpha = ln((1 − err) / err)
        raise NotImplementedError("TODO 4：ε、α")
        # TODO 5（程式）：更新權重
        #   - 錯的乘上 exp(alpha)、對的不變：w * np.exp(alpha * miss)
        #   - 正規化成總和 1
        raise NotImplementedError("TODO 5：權重更新")
        return err, alpha, w_new

    def fit(self, X: ArrayLike, y: ArrayLike) -> Self:
        X, y = np.asarray(X, dtype=float), np.asarray(y)
        self.classes_ = np.unique(y)
        s = np.where(y == self.classes_[1], 1, -1)
        w = np.full(len(s), 1 / len(s))
        self.stumps_, self.alphas_, self.weights_history_, self.errors_ = [], [], [], []
        for t in range(self.n_estimators):
            stump = WeightedStump().fit(X, s, w)
            miss = stump.predict(X) != s
            if not miss.any():  # 完全分對：給一票就停（sklearn 也是）
                self.stumps_.append(stump)
                self.alphas_.append(1.0)
                self.weights_history_.append(w)
                self.errors_.append(0.0)
                break
            err, alpha, w_new = self._boost_round(w, miss)
            if err >= 0.5:  # 比亂猜還差：停
                break
            self.stumps_.append(stump)
            self.alphas_.append(alpha)
            self.weights_history_.append(w)
            self.errors_.append(err)
            w = w_new
        self.alphas_ = np.array(self.alphas_)
        self.weights_history_ = np.array(self.weights_history_)
        self.errors_ = np.array(self.errors_)
        return self

    def decision_function(self, X: ArrayLike) -> np.ndarray:
        """(n,)：F(x) = Σ_t α_t h_t(x)。"""
        X = np.asarray(X, dtype=float)
        F = np.zeros(len(X))
        # TODO 6（程式）：加權投票
        #   - for stump, alpha in zip(self.stumps_, self.alphas_): F += alpha * stump.predict(X)
        raise NotImplementedError("TODO 6：加權投票")
        return F

    def staged_decision_function(self, X: ArrayLike) -> np.ndarray:
        """(T, n)：第 t 列 = 前 t+1 輪的 F。"""
        X = np.asarray(X, dtype=float)
        H = np.array([stump.predict(X) for stump in self.stumps_])  # (T, n)
        # TODO 7（程式）：每一輪的累計分數
        #   - 每一列乘上自己的 alpha（alphas_[:, None] 廣播），再沿著「輪」那一軸累加
        raise NotImplementedError("TODO 7：staged")

    def predict(self, X: ArrayLike) -> np.ndarray:
        return np.where(self.decision_function(X) > 0, self.classes_[1], self.classes_[0])

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

for T in [1, 10, 50, 200]:
    mine = AdaBoost(n_estimators=T).fit(Xtr, ytr)
    sk = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=T, random_state=0).fit(Xtr, ytr)
    F_mine, F_sk = mine.decision_function(Xte), sk.decision_function(Xte)
    ratio = F_mine / F_sk  # sklearn 的 decision_function 是 F 乘上一個正的常數（正規化過）
    print(
        f"T={T:<4} ✅ α 與 sklearn estimator_weights_ 最大差異 {np.abs(mine.alphas_ - sk.estimator_weights_[: len(mine.alphas_)]).max():.1e}（應 < 1e-10）",
        f"；F ÷ sklearn decision_function 的比值：最小 {ratio.min():.6f}、最大 {ratio.max():.6f}（應相同 → 成正比）",
        f"；預測一致 {np.mean(mine.predict(Xte) == sk.predict(Xte)):.3f}；test acc {np.mean(mine.predict(Xte) == yte):.4f}",
    )
ada200 = AdaBoost(n_estimators=200).fit(Xtr, ytr)
st_F = ada200.staged_decision_function(Xte)
print("✅ staged 的最後一列 = decision_function：", np.allclose(st_F[-1], ada200.decision_function(Xte)))

# 每一輪「更新後」的權重下，h_t 的錯誤率應該剛好 0.5
w_next = ada200.weights_history_[1:]
miss_t = np.array([st.predict(Xtr) != s_tr for st in ada200.stumps_[:-1]])
print("✅ 更新後，上一個 stump 在新權重下的錯誤率：", np.round((w_next * miss_t).sum(axis=1)[:5], 6), "...（應都是 0.5）")

In [ ]:
# TODO 8（問答）α 從哪裡來：指數損失
#
# AdaBoost 可以看成「一次加一個 stump，讓指數損失 L(F) = Σ_i exp(−y_i F(x_i)) 下降最多」。
# 假設已經有 F_{t−1}，現在要加 α·h（h 已經挑好），令 w_i ∝ exp(−y_i F_{t−1}(x_i))（目前每一筆的損失 = 它的權重）。
#
# (1) (a) 證明 L(F_{t−1} + α h) ∝ Σ_i w_i exp(−α y_i h(x_i))。（前置複習 2：e^{a+b} = e^a e^b）
#     (b) y_i h(x_i) 只有兩種值：分對 = +1、分錯 = −1。把 (a) 拆成「分對的 Σ w_i · e^{−α}」+「分錯的 Σ w_i · e^{α}」，
#         用 ε（分錯的權重總和）寫出來。
#     (c) 用前置複習 3 對 α 求最小值，得到 α = ½ ln((1 − ε)/ε)。跟程式裡的 α 差幾倍？為什麼預測結果一樣？
# 答：
#
# (2) 權重 w_i ∝ exp(−y_i F(x_i))：
#     (a) 一筆樣本的權重大，代表它的 margin y_i F(x_i) 怎樣？
#     (b) 為什麼「更新後 h_t 的錯誤率剛好 0.5」？（提示：分錯的總權重 ε 乘 (1−ε)/ε = 1 − ε，分對的仍是 1 − ε）
#         這對下一個 stump 有什麼意義？
# 答：
#
# (3) 指數損失 vs 第三章的 log loss（都是 margin z = yF 的函數）：z 很負（嚴重分錯）時，e^{−z} 和 log(1 + e^{−z}) 各怎麼長？
#     哪一個對「標錯的資料」比較敏感？（3C 的實驗 3 會驗證）
# 答：

### 3C. 實驗：輪數與 margin、權重、標錯的資料、基底分類器與學習率

這一節用手寫的 `AdaBoost`（實驗 1、2）與 sklearn（實驗 3、4，比較快）。

#### 本節名詞

| 名詞 | 定義 |
| --- | --- |
| 訓練誤差／測試誤差 | $\text{sign}(F_t)$ 在訓練集／測試集上的錯誤率，對每個 $t$ 各算一次 |
| 指數損失 | $\frac{1}{n}\sum_i e^{-y_i F_t(x_i)}$（訓練集） |
| 正規化 margin | $y_i F_t(x_i) / \sum_{s \le t} \alpha_s \in [-1, 1]$ |
| 標籤雜訊 | 故意把訓練集 10% 的標籤翻轉（測試集不動） |
| `learning_rate` $\nu$ | sklearn 的 $\alpha_t$ 會乘上 $\nu$（$0 < \nu \le 1$）：每一步走小一點 |

In [ ]:
# 實驗 1：輪數。訓練誤差、測試誤差、指數損失，以及 margin 的分布
ada1000 = AdaBoost(n_estimators=1000).fit(Xtr, ytr)
s_te = np.where(yte == 1, 1, -1)
F_tr_st = ada1000.staged_decision_function(Xtr)
F_te_st = ada1000.staged_decision_function(Xte)
T_used = len(ada1000.alphas_)
rounds = np.arange(1, T_used + 1)
curve = pd.DataFrame(
    {
        "train error": (np.sign(F_tr_st) != s_tr).mean(axis=1),
        "test error": (np.sign(F_te_st) != s_te).mean(axis=1),
        "train exp loss": np.exp(-s_tr * F_tr_st / 2).mean(axis=1),  # /2：換成 TODO 8 的 ½ 版本 α
        "weighted error ε_t": ada1000.errors_,
    },
    index=rounds,
)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].plot(rounds, curve["train error"], label="train error")
axes[0].plot(rounds, curve["test error"], label="test error")
axes[0].plot(rounds, curve["train exp loss"], label="train exp loss (upper bound)")
axes[0].set_xscale("log")
axes[0].set_xlabel("rounds t (log)")
axes[0].set_ylim(0, 0.5)
axes[0].legend()
axes[0].set_title("Errors vs rounds")
axes[1].plot(rounds, curve["weighted error ε_t"], ".", ms=3)
axes[1].set_xscale("log")
axes[1].axhline(0.5, color="r", ls=":")
axes[1].set_xlabel("rounds t (log)")
axes[1].set_ylabel("ε_t")
axes[1].set_title("Weighted error of each new stump")
for t in [5, 50, 1000]:
    if t <= T_used:
        margins = s_tr * F_tr_st[t - 1] / ada1000.alphas_[:t].sum()
        axes[2].plot(np.sort(margins), np.linspace(0, 1, len(margins)), label=f"t = {t}")
axes[2].axvline(0, color="k", lw=0.8)
axes[2].set_xlabel("normalized margin y·F / Σα (train)")
axes[2].set_ylabel("cumulative fraction")
axes[2].legend()
axes[2].set_title("Margin distribution")
fig.tight_layout()
plt.show()
first_zero = int(np.argmax(curve["train error"].to_numpy() == 0)) + 1 if (curve["train error"] == 0).any() else None
print("訓練誤差第一次 = 0 的輪數：", first_zero)
display(curve.loc[[t for t in [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000] if t <= T_used]].round(4))

In [ ]:
# 實驗 2：權重集中在哪？
W_hist = ada1000.weights_history_
t_show = min(100, T_used) - 1
w_t = W_hist[t_show]
top = np.argsort(w_t)[::-1][:10]
margin_t = s_tr * F_tr_st[t_show - 1] / ada1000.alphas_[:t_show].sum()
print(f"第 {t_show + 1} 輪開始時，權重最大的 10 筆佔了全部權重的 {w_t[top].sum():.1%}（等權重時是 {10 / len(w_t):.1%}）")
display(pd.DataFrame({"weight": w_t[top], "label": ytr[top], "margin before this round": margin_t[top]}, index=X_train.index[top]).round(4))

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z2 = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(Xtr))  # 只為了「看」
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, (t_i, title) in zip(axes, [(0, "round 1 (equal weights)"), (t_show, f"round {t_show + 1}")]):
    ax.scatter(Z2[:, 0], Z2[:, 1], c=ytr, cmap="coolwarm", s=4000 * W_hist[t_i], alpha=0.6, edgecolors="k", linewidths=0.3)
    ax.set_title(f"Sample weights at {title} (size ∝ weight)")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
plt.show()

In [ ]:
# TODO 9（問答）輪數、margin、權重
#
# (1) 實驗 1 左圖與表：
#     (a) 訓練誤差在第幾輪變成 0？測試誤差在那之後還有下降嗎？
#     (b) 指數損失一直在降嗎？它跟訓練誤差的關係是什麼？（前置複習 4：上界）
#     (c) 「訓練誤差已經是 0，再加 stump 卻還在改善測試誤差」—— 這跟第五章「樹長太深就過擬合」的經驗相反。
#         右圖：t 越大，margin 的分布怎麼變？最小的 margin 有變大嗎？用第八章 SVM「margin 越大越好」的觀點解釋。
#     (d) 那 AdaBoost 會過擬合嗎？看 1000 輪時的測試誤差（第 5 節會用 CV 選輪數）。
# 答：
#
# (2) 中圖：每個新 stump 的加權錯誤率 ε_t 隨 t 怎麼變？為什麼越來越接近 0.5？（TODO 8 (2)(b)）
#     ε_t 接近 0.5 時 α_t 接近多少？這些後期的 stump 還有貢獻嗎？
# 答：
#
# (3) 實驗 2：
#     (a) 第 100 輪時，權重最大的 10 筆佔了多少？它們的 margin 有什麼特徵？
#     (b) 在 PCA 圖上，大權重的點落在哪裡？（兩類交界？還是某一類的中心？）
#     (c) 如果其中有幾筆其實是「標錯」的，AdaBoost 會怎麼對待它們？
# 答：

In [ ]:
# 實驗 3：標籤雜訊。把訓練集 10% 的標籤翻轉，測試集不動
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng_n = np.random.default_rng(0)
flip = rng_n.choice(len(ytr), size=int(0.1 * len(ytr)), replace=False)
y_noisy = ytr.copy()
y_noisy[flip] = 1 - y_noisy[flip]

models = {
    "AdaBoost stumps, T=500": lambda: AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=500, random_state=0),
    "AdaBoost stumps, T=50": lambda: AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, random_state=0),
    "Random Forest 300": lambda: RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1),
    "LogReg (standardized)": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}
rows = []
for name, make in models.items():
    clean, noisy = make().fit(Xtr, ytr), make().fit(Xtr, y_noisy)
    rows.append({"model": name, "test acc (clean labels)": clean.score(Xte, yte), "test acc (10% flipped)": noisy.score(Xte, yte)})
noise_tbl = pd.DataFrame(rows).set_index("model")
noise_tbl["drop"] = noise_tbl.iloc[:, 0] - noise_tbl.iloc[:, 1]
display(noise_tbl.round(4))

ada_noisy = AdaBoost(n_estimators=300).fit(Xtr, y_noisy)
share = ada_noisy.weights_history_[:, flip].sum(axis=1)
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.arange(1, len(share) + 1), share)
ax.axhline(0.1, color="gray", ls=":", label="share of samples (10%)")
ax.set_xscale("log")
ax.set_xlabel("round t (log)")
ax.set_ylabel("total weight on flipped samples")
ax.legend()
ax.set_title("Where does AdaBoost put its attention under label noise?")
plt.show()
print(f"第 {len(share)} 輪時，翻轉的 {len(flip)} 筆佔了 {share[-1]:.1%} 的權重")

In [ ]:
# 實驗 4：基底分類器的深度 × 學習率（重複 5-fold CV，T = 200）
rows = []
for depth in [1, 2, 3]:
    for lr in [0.1, 0.5, 1.0]:
        s = cross_validate(AdaBoostClassifier(DecisionTreeClassifier(max_depth=depth), n_estimators=200, learning_rate=lr, random_state=0), Xtr, ytr, cv=cv, scoring="accuracy", n_jobs=-1)
        rows.append({"max_depth": depth, "learning_rate": lr, "CV acc": s["test_score"].mean(), "std": s["test_score"].std()})
dl = pd.DataFrame(rows)
display(dl.pivot_table(index="max_depth", columns="learning_rate", values="CV acc").round(4))

In [ ]:
# TODO 10（問答）雜訊、深度、學習率
#
# (1) 實驗 3：
#     (a) 四個模型在 10% 標籤翻轉之後各掉了多少？哪一個掉最多？T = 500 和 T = 50 的 AdaBoost 誰掉得多？
#     (b) 右圖：翻轉的樣本只佔 10%，卻吃掉多少比例的權重？用指數損失 e^{−z}（z 很負時）解釋。
#     (c) 那 Random Forest 為什麼比較不怕？（提示：bagging 裡每棵樹「平等地」看資料，沒有人會專門盯著某幾筆）
#     (d) 實務上要怎麼讓 boosting 對雜訊比較穩？（提示：輪數、學習率、換一個損失函數 —— 第 12 章 GBDT 可以用 log loss）
# 答：
#
# (2) 實驗 4：
#     (a) max_depth = 1、2、3 哪個最好？差距大嗎？
#     (b) 基底分類器越強（越深），boosting 能「修正」的空間越大還是越小？太強會有什麼風險？（TODO 2 (2) 的預測對嗎？）
#     (c) learning_rate 小，同樣的 T 下表現怎麼變？「學習率小 + 輪數多」通常比較好，代價是什麼？
# 答：

---
## 4. 套件實作：sklearn `AdaBoostClassifier`

#### 本節名詞

| 名詞 | 意思 |
| --- | --- |
| `estimator` | 基底分類器，預設 `DecisionTreeClassifier(max_depth=1)` |
| `estimator_weights_` | 每一輪的 $\alpha_t$（已乘上 `learning_rate`） |
| `estimator_errors_` | 每一輪的 $\varepsilon_t$ |
| `decision_function` | $F(x)$ 乘上一個正的常數（依 $\sum_t \alpha_t$ 正規化過；sign 與排序都一樣） |
| `staged_predict`／`staged_decision_function` | 每一輪的預測（一次 fit 就能畫出整條曲線） |
| `predict_proba` | 由 decision_function 換算的機率（不是校準過的機率） |
| `feature_importances_` | 每個 stump 的 impurity importance 依 $\alpha_t$ 加權平均 |

#### 程式結構圖

```
ada_pipe = AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=50, learning_rate=1.0)
├── estimators_（T 個 stump）、estimator_weights_ (T,)、estimator_errors_ (T,)
└── 參數名：n_estimators、learning_rate、estimator__max_depth
```
（不需要前處理，所以不用 Pipeline。）

In [ ]:
ada_default = AdaBoostClassifier(random_state=0).fit(Xtr, ytr)
print("預設：estimator =", ada_default.estimator_, "、n_estimators =", ada_default.n_estimators, "、learning_rate =", ada_default.learning_rate)
baseline.loc["AdaBoost default (T=50)"] = cv_summary(AdaBoostClassifier(random_state=0))
baseline.loc["AdaBoost T=200"] = cv_summary(AdaBoostClassifier(n_estimators=200, random_state=0))
display(baseline)

imp = pd.Series(ada_default.feature_importances_, index=FEATURES).sort_values(ascending=False)
used = pd.Series([FEATURES[e.tree_.feature[0]] for e in ada_default.estimators_]).value_counts()
print("50 個 stump 用到的特徵（次數）：")
display(used.head(10).to_frame("count").T)
print("feature_importances_ 前 5：", imp.head(5).round(4).to_dict())

In [ ]:
# TODO 11（問答）
#
# (1) 50 個 stump 用到幾個不同的特徵？有沒有特徵被用了很多次？
#     同一個特徵被不同的 stump 用了好幾次（門檻不同），加起來在那個特徵上形成什麼形狀的函數？
#     （提示：F(x) 是很多個 α·(±1 階梯) 的和 → 在單一特徵上是「多段的階梯」）
# 答：
#
# (2) AdaBoost + stump 的 F(x) = Σ_t α_t h_t(x)，每個 h_t 只看一個特徵。
#     所以 F 可以寫成 f_1(x_1) + f_2(x_2) + … + f_30(x_30)（每個特徵各自一個函數相加，叫「加法模型」）。
#     (a) 這種模型能表達「特徵 A 的效果取決於特徵 B」（交互作用）嗎？第五章的決策樹可以嗎？
#     (b) 如果改用深度 2 的樹當基底分類器呢？
# 答：
#
# (3) 跟第三章的邏輯回歸比：兩者的 CV accuracy、AUC 差多少？
#     邏輯回歸也是「每個特徵各自一個函數相加」（w_j x_j），差別在函數的形狀：一個是直線，一個是階梯。
#     在這份資料上，彈性比較大的階梯有幫到忙嗎？
# 答：

---
## 5. 調參

全程只用 `X_train`，重複 5-fold CV。

| 類別 | 超參數 | 說明 |
| --- | --- | --- |
| 容量 | `n_estimators` $T$ | 越多越複雜（但不像深樹那麼容易過擬合，TODO 9） |
| 最佳化 | `learning_rate` $\nu$ | 每一步走多遠；跟 $T$ 一起調：$\nu$ 小需要 $T$ 大 |
| 容量 | `estimator__max_depth` | 基底分類器的強度 |

**選輪數的捷徑**：$T$ 不用一個一個 fit —— 一次 fit $T_\text{max}$ 輪，`staged_predict` 就給出每一輪的預測。每個 fold 各算一條「輪數 → accuracy」的曲線，再平均（TODO 12）。

#### 本節名詞

- **staged CV 曲線**：$(\text{folds}, T)$ 的分數矩陣，對 fold 那一軸取平均與標準差。
- **1-SE（輪數）**：門檻 = 最佳輪數的平均 − 標準差；在門檻內挑**最少**的輪數（模型最小、預測最快）。

#### 最小例子：TODO 12（staged CV 曲線與 1-SE）

3 個 fold、4 輪的 accuracy 矩陣 `S = [[0.90, 0.93, 0.95, 0.94], [0.88, 0.94, 0.94, 0.95], [0.91, 0.92, 0.96, 0.95]]`。

1. **正確答案**：逐輪算平均與標準差；最佳是第 3 輪（平均 0.95）；門檻 0.95 − 0.0082 = 0.9418；平均 ≥ 門檻的最小輪數 = 第 3 輪（第 2 輪平均 0.93 不夠）。
2. **積木**：`S.mean(axis=0)`、`S.std(axis=0)`、`np.argmax`、`np.flatnonzero(mean >= thr)[0]`。
3. **留給你**：組出 best 與 1-SE 的輪數（**從 1 開始數**）。

In [ ]:
# 最小例子：TODO 12（staged CV 曲線與 1-SE）
S_mini = np.array([[0.90, 0.93, 0.95, 0.94], [0.88, 0.94, 0.94, 0.95], [0.91, 0.92, 0.96, 0.95]])  # (folds, T)

# ---- 1. 正確答案：逐輪 ----
means = [float(np.mean(S_mini[:, t])) for t in range(4)]
stds = [float(np.std(S_mini[:, t])) for t in range(4)]
t_best = max(range(4), key=lambda t: means[t])
thr = means[t_best] - stds[t_best]
t_1se = min(t for t in range(4) if means[t] >= thr)
print("平均", np.round(means, 4), " 標準差", np.round(stds, 4))
print(f"正確答案：最佳第 {t_best + 1} 輪、門檻 {thr:.4f}、1-SE 第 {t_1se + 1} 輪")

# ---- 2. 積木 ----
print("\nS.mean(axis=0) =", S_mini.mean(axis=0).round(4), "← 對 fold 平均，每輪一個數")
print("np.flatnonzero([F, T, T, T]) =", np.flatnonzero([False, True, True, True]), "← [0] 就是第一個 True 的位置")

# ---- 3. 留給你 ----
# 問題：best_T = argmax(mean) + 1；thr = mean[best] − std[best]；one_se_T = 第一個 mean ≥ thr 的位置 + 1

In [ ]:
# staged CV：learning_rate × max_depth，每組一次 fit T_MAX 輪，取每一輪的 accuracy
T_MAX = 500
cv_small = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
staged = {}
for depth in [1, 2]:
    for lr in [0.1, 0.5, 1.0]:
        S = []
        for tr_i, va_i in cv_small.split(Xtr, ytr):
            m = AdaBoostClassifier(DecisionTreeClassifier(max_depth=depth), n_estimators=T_MAX, learning_rate=lr, random_state=0).fit(Xtr[tr_i], ytr[tr_i])
            acc = [np.mean(p == ytr[va_i]) for p in m.staged_predict(Xtr[va_i])]
            acc += [acc[-1]] * (T_MAX - len(acc))  # 提早停下時補齊
            S.append(acc)
        staged[(depth, lr)] = np.array(S)  # (folds, T_MAX)

rows = []
for (depth, lr), S in staged.items():
    mean, std = S.mean(axis=0), S.std(axis=0)
    # TODO 12（程式）：最佳輪數與 1-SE 輪數（從 1 開始數）
    #   - best = np.argmax(mean)；best_T = best + 1
    #   - thr = mean[best] − std[best]；one_se_T = np.flatnonzero(mean >= thr)[0] + 1
    raise NotImplementedError("TODO 12：staged 1-SE")
    rows.append({"max_depth": depth, "learning_rate": lr, "best T": best_T, "best CV acc": mean[best], "std": std[best], "1-SE T": one_se_T, "1-SE CV acc": mean[one_se_T - 1]})
stage_tbl = pd.DataFrame(rows)
display(stage_tbl.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
for (depth, lr), S in staged.items():
    ax.plot(np.arange(1, T_MAX + 1), S.mean(axis=0), label=f"depth {depth}, lr {lr}")
ax.set_xscale("log")
ax.set_xlabel("rounds (log)")
ax.set_ylabel("CV accuracy")
ax.set_ylim(0.9, 1.0)
ax.legend(fontsize=8)
ax.set_title("Staged CV accuracy")
plt.show()

best_row = stage_tbl.loc[stage_tbl["best CV acc"].idxmax()]
choice = stage_tbl.loc[stage_tbl["1-SE CV acc"] >= best_row["best CV acc"] - best_row["std"]].sort_values(["max_depth", "1-SE T"]).iloc[0]
print("CV 最佳：", best_row.to_dict())
print("1-SE（depth 小、T 少優先）：", choice.to_dict())

In [ ]:
# TODO 13（問答）
#
# (1) staged 曲線：
#     (a) learning_rate = 1 的曲線在哪一輪附近到頂？0.1 的呢？到頂之後有明顯下降（過擬合）嗎？
#     (b) 每一組的 best T 與 1-SE T 差多少輪？差多少 accuracy？
# 答：
#
# (2) 最後選的組合是什麼？跟 CV 最佳差多少？「簡單」在這裡換到了什麼？（stump 的個數 = 預測時要問幾個問題）
# 答：
#
# (3) 用 staged_predict 選輪數，比 GridSearchCV 試很多個 n_estimators 省多少次 fit？
#     （第 10 章 RF 用 OOB 省時間，這裡用 staged；第 12 章 GBDT 會用 early stopping —— 都是同一個想法）
# 答：

---
## 6. 測試集評估

測試集只評估一次，成績寫進 **`results/breast_cancer.csv`** —— 跟第三章邏輯回歸的紀錄放在一起比。

- 對照組（未調參）：**Random Forest**（第十章，300 棵）。邏輯回歸的成績第三章已經記錄過，`compare` 會一起列出來。
- 測試集 114 筆（1 筆 ≈ 0.88%）。

#### 指標

- accuracy、precision／recall／F1（以良性 = 1 為正類，跟第三章一樣）、ROC AUC，以及 **`recall_malignant`**（惡性被抓到的比例）。

In [ ]:
from learning_map.metrics import classification_metrics
from learning_map.results import compare, log_result

BENCH = "breast_cancer"
final = AdaBoostClassifier(DecisionTreeClassifier(max_depth=int(choice["max_depth"])), n_estimators=int(choice["1-SE T"]), learning_rate=float(choice["learning_rate"]), random_state=0).fit(Xtr, ytr)
best_cv = AdaBoostClassifier(DecisionTreeClassifier(max_depth=int(best_row["max_depth"])), n_estimators=int(best_row["best T"]), learning_rate=float(best_row["learning_rate"]), random_state=0).fit(Xtr, ytr)
rf = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
ada_scratch = AdaBoost(n_estimators=200).fit(Xtr, ytr)
F_s = ada_scratch.decision_function(Xte)

# 名稱: (預測, 機率或分數（用來算 AUC）, 參數)
eval_models = {
    "Single stump (sklearn)": (sk_st.predict(Xte), sk_st.predict_proba(Xte)[:, 1], {"max_depth": 1}),
    "AdaBoost stumps T=200 (scratch)": (ada_scratch.predict(Xte), 1 / (1 + np.exp(-F_s)), {"n_estimators": 200}),
    "AdaBoost default T=50 (sklearn)": (ada_default.predict(Xte), ada_default.predict_proba(Xte)[:, 1], {"n_estimators": 50}),
    "AdaBoost CV-best (sklearn)": (best_cv.predict(Xte), best_cv.predict_proba(Xte)[:, 1], {"max_depth": int(best_row["max_depth"]), "learning_rate": float(best_row["learning_rate"]), "n_estimators": int(best_row["best T"])}),
    "AdaBoost 1-SE (sklearn)": (final.predict(Xte), final.predict_proba(Xte)[:, 1], {"max_depth": int(choice["max_depth"]), "learning_rate": float(choice["learning_rate"]), "n_estimators": int(choice["1-SE T"])}),
    "Random Forest 300 (sklearn, ref)": (rf.predict(Xte), rf.predict_proba(Xte)[:, 1], {"n_estimators": 300}),
}
test_scores = {}
for name, (pred, prob, params) in eval_models.items():
    m = classification_metrics(y_test, pred, prob)
    m["recall_malignant"] = round(recall_score(y_test, pred, pos_label=0), 4)
    test_scores[name] = m
    log_result(BENCH, name, m, **params)
display(pd.DataFrame(test_scores).T)
compare(BENCH, "roc_auc")

---
## 7. 診斷

#### 本節名詞

- **混淆矩陣**：列 = 真實（0 惡性、1 良性）、欄 = 預測。
- **測試集的 margin**：$y \cdot$ `decision_function`$(x)$（sklearn 已經正規化過）；負的就是分錯，接近 0 就是「差點分錯」。
- **第三章的錯誤樣本**：第三章邏輯回歸在同一個測試集上錯的那幾筆（如果那裡記下了 index，可以對照；這裡直接重訓一個邏輯回歸比較）。

In [ ]:
from learning_map.metrics import plot_confusion_matrix

pred_te = final.predict(Xte)
plot_confusion_matrix(y_test, pred_te, labels=["malignant", "benign"])
plt.title("AdaBoost (1-SE) on test")
plt.show()

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
margin_te = np.where(yte == 1, 1, -1) * final.decision_function(Xte)
diag = pd.DataFrame(
    {"true": yte, "AdaBoost pred": pred_te, "AdaBoost margin": margin_te, "LogReg pred": logreg.predict(Xte), "LogReg P(benign)": logreg.predict_proba(Xte)[:, 1]},
    index=X_test.index,
)
print("AdaBoost 或邏輯回歸分錯、或 AdaBoost margin 最小的幾筆：")
show = diag[(diag["AdaBoost pred"] != diag["true"]) | (diag["LogReg pred"] != diag["true"])]
display(pd.concat([show, diag.nsmallest(5, "AdaBoost margin")]).drop_duplicates().sort_values("AdaBoost margin").round(4))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(margin_te, bins=30)
ax.axvline(0, color="r", ls="--")
ax.set_xlabel("normalized margin on test")
ax.set_title("Test margins (AdaBoost 1-SE)")
plt.show()

In [ ]:
# TODO 14（問答，本章總結）
#
# (1) 錯在哪裡：
#     (a) AdaBoost 錯了幾筆？是惡性被當良性（漏診），還是反過來？哪一種在醫療上比較嚴重？
#     (b) AdaBoost 和邏輯回歸錯的是同一批樣本嗎？如果大致相同，代表問題比較可能出在模型還是資料？
#     (c) margin 接近 0 的那幾筆，可以當作「需要人工複查」的訊號嗎？
# 答：
#
# (2) 跟前幾章比（看第 6 節的 compare 表，包含第三章的紀錄）：
#     (a) AdaBoost、Random Forest、邏輯回歸的 test accuracy、AUC、recall_malignant 各多少？114 筆裡差幾筆？
#     (b) 比較表（填空）：
#
#         |                            | AdaBoost（stump） | Random Forest | 邏輯回歸 |
#         | bagging／boosting／都不是  |                   |               |          |
#         | 主要降低偏差還是變異       |                   |               |          |
#         | 基底模型                   |                   |               |          |
#         | 訓練能不能平行             |                   |               |          |
#         | 對標錯的資料               |                   |               |          |
#         | 需要標準化嗎               |                   |               |          |
#
#     (c) 在這份資料上，複雜的集成方法有比第三章簡單的邏輯回歸好嗎？為什麼可能沒有？（提示：第三章 EDA —— 這份資料的類別邊界大致是什麼形狀？）
# 答：

---
## 8. 本章小結

（完成所有 TODO、並經過回饋與修正之後，依 README「章末小結」填寫：問答註解轉成 markdown、結果表、核心觀念、待補強。）

### 方法與結果

Breast Cancer（train 455 / test 114，與第三章相同的切分；CV = 5-fold × 10 次）。

| 方法 | 做法 | 關鍵性質 | CV accuracy | test accuracy | test recall_malignant |
| --- | --- | --- | --- | --- | --- |
| Dummy prior | | | | | |
| 單一 stump | | | | | |
| AdaBoost（手寫，T = 200） | 加權 Gini stump + SAMME | | — | | |
| AdaBoost（sklearn，CV 最佳／1-SE） | staged CV 選輪數 | | | | |
| Random Forest（對照） | | | — | | |
| 邏輯回歸（第三章） | | | | | |

### 核心觀念

1.
2.
3.

### 待補強（之後回來可以學）

**理論**
- ☐ AdaBoost 的訓練誤差上界 $\prod_t 2\sqrt{\varepsilon_t(1 - \varepsilon_t)}$ 的推導
- ☐ Margin 理論（Schapire et al. 1998）與它的爭議（Breiman 的反例）
- ☐ AdaBoost = 指數損失的前向分段加法模型（Friedman, Hastie & Tibshirani 2000）的完整推導；Real AdaBoost、LogitBoost、Gentle AdaBoost
- ☐ 弱可學習性（weak learnability）與 boosting 的 PAC 理論

**延伸**
- ☐ 多類別：SAMME 的 $\alpha$ 多了 $\ln(K - 1)$；SAMME.R（已從 sklearn 移除）
- ☐ AdaBoost.R2（回歸）
- ☐ 對雜訊比較穩的 boosting：BrownBoost、LogitBoost；用 log loss 的梯度提升（第 12 章）
- ☐ Viola–Jones 人臉偵測：AdaBoost + Haar 特徵 + cascade（經典應用）